# Cluster effect sizes and risk-factor forest plots

Marianna Georgiou, Department of Psychiatry, Psychotherapy and Psychosomatics, Faculty of Medicine, RWTH Aachen, Aachen, Germany
**Email:** mariannag43@gmail.com
**Publication:**  Latent subtypes of longitudinal postpartum trajectories for identifying healthy women at risk for affective conditions

Compare participant-averaged raw scores between selected diagnosis/cluster groups and visualize discovery/validation risk-factor summaries.

## Inputs

Under `files/interm_files/`:

- `disc_merged_rawdata_withlabels.xlsx` or `val_merged_rawdata_withlabels.xlsx`, selected by `coh` (default `[1, 2]`). Requires Diagnosis (HC/AD/PPD), IDX=1/2, ID, and raw score columns. Discovery EPDS totals use `_y`; validation totals do not.
- `rr_cat_[1, 2]_statanalysis.xlsx` and `rr_cat_[3]_statanalysis.xlsx`.
- `cd_cont_[1, 2]_statanalysis.xlsx` and `cd_cont_[3]_statanalysis.xlsx`.

Generate both cohorts’ risk-factor summaries before running the forest-plot sections.

## Outputs

- `files/tables/{cohort}_clust_cohens.xlsx`, without row index.
- `files/plots/{cohort}_clustered_cohens_violin` in PNG/PDF/TIFF.
- `files/plots/relative_risk` and `cohensd_risk` in PNG/PDF/TIFF filenames.
- `files/plots/merged_riskfactors.tiff` and `merged_riskfactors_labeled.tiff`.

`{cohort}` is `disc` or `val`. 

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import matplotlib.patheffects as path_effects

%load_ext dotenv
%dotenv
%load_ext autoreload
%autoreload 2
from pathlib import Path

BASE_DIR = Path.cwd()
INPUT_PATH = BASE_DIR / "files" / "inputs"
OUTPUT_FILES = BASE_DIR / "files" / "interm_files"
OUTPUT_PLOTS = BASE_DIR / "files" / "plots"
OUTPUT_TABLES = BASE_DIR / "files" / "tables"
for directory in (OUTPUT_FILES, OUTPUT_PLOTS, OUTPUT_TABLES):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
coh = [1, 2]


In [ ]:
def set_global_plot_style():

    """Apply the figure typography and Seaborn style."""
    sns.set_context("paper", font_scale=1.2)

    plt.rcParams['font.family'] = 'Arial'
    plt.rc('font', size=9)
    plt.rc('axes', titlesize=9)
    plt.rc('axes', labelsize=9)
    plt.rc('xtick', labelsize=9)
    plt.rc('ytick', labelsize=9)
    plt.rc('legend', fontsize=9)
    plt.rc('figure', titlesize=9)

set_global_plot_style()

np.set_printoptions(precision=3, suppress=True)

colors = {
    ('HC', 0): '#0072B2',
    ('HC', 1): '#003F5C',
    ('HC', 2): '#56B4E9',
    ('AD', 0): '#D55E00',
    ('AD', 1): '#B35806',
    ('AD', 2): '#E69F00',
    ('PPD', 0): '#009E73',
    ('PPD', 1): '#00441B',
    ('PPD', 2): '#66C2A5',
}

custom_palette = {
    'HC': colors[('HC', 0)],
    'AD': colors[('AD', 0)],
    'PPD': colors[('PPD', 0)]
}


## 2. Prepare labeled raw trajectories

Form groups from Diagnosis and IDX, reshape to long format, and leave MPAS T0 missing. Input labels must match the fixed group names used below.

In [ ]:
if coh == [3]:
    input_labeled_raw_disc = OUTPUT_FILES / "val_merged_rawdata_withlabels.xlsx"
    labeled_raw = pd.read_excel(input_labeled_raw_disc)
elif coh == [1, 2]:
    input_labeled_raw_disc = OUTPUT_FILES / "disc_merged_rawdata_withlabels.xlsx"
    labeled_raw = pd.read_excel(input_labeled_raw_disc)

labeled_raw["MPAS T0"] = np.nan
labeled_raw['Group'] = labeled_raw['Diagnosis'] + ' C' + (labeled_raw['IDX']).astype(str)
labeled_raw['Group'].value_counts()


In [ ]:
stress_df = labeled_raw.melt(
    id_vars=['ID', 'Group'],
    value_vars=['Stress T0', 'Stress T1', 'Stress T2', 'Stress T3', 'Stress T4'],
    var_name='Time',
    value_name='Stress'
)

mood_df = labeled_raw.melt(
    id_vars=['ID', 'Group'],
    value_vars=['Mood T0', 'Mood T1', 'Mood T2', 'Mood T3', 'Mood T4'],
    var_name='Time',
    value_name='Mood'
)

mpas_df = labeled_raw.melt(
    id_vars=['ID', 'Group'],
    value_vars=['MPAS T0', 'MPAS T1', 'MPAS T2', 'MPAS T3', 'MPAS T4'],
    var_name='Time',
    value_name='MPAS'
)

anhedonia_df = labeled_raw.melt(
    id_vars=['ID', 'Group'],
    value_vars=['Anhedonie_t0', 'Anhedonie_t1', 'Anhedonie_t2', 'Anhedonie_t3', 'Anhedonie_t4'],
    var_name='Time',
    value_name='Anhedonia'
)

anxiety_df = labeled_raw.melt(
    id_vars=['ID', 'Group'],
    value_vars=['Anxiety_t0', 'Anxiety_t1', 'Anxiety_t2', 'Anxiety_t3', 'Anxiety_t4'],
    var_name='Time',
    value_name='Anxiety'
)

depression_df = labeled_raw.melt(
    id_vars=['ID', 'Group'],
    value_vars=['Depression_t0', 'Depression_t1', 'Depression_t2', 'Depression_t3', 'Depression_t4'],
    var_name='Time',
    value_name='Depression'
)

if coh==[3]:
    epds_df = labeled_raw.melt(
        id_vars=['ID', 'Group'],
        value_vars=['EPDStotal_t0', 'EPDStotal_t1', 'EPDStotal_t2', 'EPDStotal_t3', 'EPDStotal_t4'],
        var_name='Time',
        value_name='EPDS'
    )
else:
    epds_df = labeled_raw.melt(
        id_vars=['ID', 'Group'],
        value_vars=['EPDStotal_t0_y', 'EPDStotal_t1_y', 'EPDStotal_t2_y', 'EPDStotal_t3_y', 'EPDStotal_t4_y'],
        var_name='Time',
        value_name='EPDS'
    )
epds_df


## 3. Participant-level cluster effect sizes

Average available time points per participant/group and compare HC C1–HC C2, HC C2–AD C2, HC C2–PPD C2, and AD C2–PPD C2.

Cohen’s d is `(mean of group 2 − mean of group 1) / pooled sample SD`. Independently resample observations within each group 10,000 times, resetting a NumPy generator to seed 42 for each call, and use the 2.5th–97.5th percentiles for the interval. Empty groups, insufficient nonmissing data, and zero pooled variance are not fully guarded against.

In [ ]:
from itertools import combinations

def cohen_d_bootstrap(group1, group2, n_bootstrap=10000, alpha=0.05, random_state=42):

    """Return group-2 minus group-1 Cohen’s d and a percentile bootstrap interval."""
    rng = np.random.default_rng(random_state)

    mean1, mean2 = np.mean(group1), np.mean(group2)
    std1, std2 = np.std(group1, ddof=1), np.std(group2, ddof=1)
    pooled_std = np.sqrt(((len(group1) - 1)*std1**2 + (len(group2) - 1)*std2**2) / (len(group1) + len(group2) - 2))
    # Positive values indicate a higher mean in group 2.
    mean_diff = mean2 - mean1
    cohen_d = mean_diff / pooled_std

    boot_d = []
    for _ in range(n_bootstrap):
        sample1 = rng.choice(group1, size=len(group1), replace=True)
        sample2 = rng.choice(group2, size=len(group2), replace=True)
        m1, m2 = np.mean(sample1), np.mean(sample2)
        s1, s2 = np.std(sample1, ddof=1), np.std(sample2, ddof=1)
        pooled = np.sqrt(((len(sample1) - 1)*s1**2 + (len(sample2) - 1)*s2**2) / (len(sample1) + len(sample2) - 2))
        d = (m2 - m1) / pooled
        boot_d.append(d)

    ci_lower = np.percentile(boot_d, 100 * alpha / 2)
    ci_upper = np.percentile(boot_d, 100 * (1 - alpha / 2))

    return cohen_d, ci_lower, ci_upper

def average_per_participant(df, value_col):

    """Average available time points within participant and group."""
    return df.groupby(['ID', 'Group'])[value_col].mean().reset_index()

def compare_diagnoses(df_avg, value_col):
    """Calculate effect sizes for the fixed group comparisons."""
    print(value_col)
    results = []

    selected_pairs = [
        ('HC C1', 'HC C2'),
        ('HC C2', 'AD C2'),
        ('HC C2', 'PPD C2'),
        ('AD C2', 'PPD C2')
    ]

    results = []
    for g1, g2 in selected_pairs:
        group1 = df_avg[df_avg['Group'] == g1][value_col].values
        group2 = df_avg[df_avg['Group'] == g2][value_col].values

        cohen_d, ci_lower, ci_upper = cohen_d_bootstrap(group1, group2)

        results.append({
            'Group1': g1,
            'Group2': g2,
            'Cohens_d': cohen_d,
            'CI_lower': ci_lower,
            'CI_upper': ci_upper
        })

    return pd.DataFrame(results)


In [ ]:
stress_avg = average_per_participant(stress_df, 'Stress')
stress_results = compare_diagnoses(stress_avg, 'Stress')
stress_results['Symptom'] = 'Stress'
print(stress_results)

mood_avg = average_per_participant(mood_df, 'Mood')
mood_results = compare_diagnoses(mood_avg, 'Mood')
mood_results['Symptom'] = 'Mood'
print(mood_results)

mpas_avg = average_per_participant(mpas_df, 'MPAS')
mpas_results = compare_diagnoses(mpas_avg, 'MPAS')
mpas_results['Symptom'] = 'MPAS'
print(mpas_results)

anh_avg = average_per_participant(anhedonia_df, 'Anhedonia')
anh_results = compare_diagnoses(anh_avg, 'Anhedonia')
anh_results['Symptom'] = 'Anhedonia'
print(anh_results)

dep_avg = average_per_participant(depression_df, 'Depression')
dep_results = compare_diagnoses(dep_avg, 'Depression')
dep_results['Symptom'] = 'Depression'
print(dep_results)

anx_avg = average_per_participant(anxiety_df, 'Anxiety')
anx_results = compare_diagnoses(anx_avg, 'Anxiety')
anx_results['Symptom'] = 'Anxiety'
print(anx_results)

epds_avg = average_per_participant(epds_df, 'EPDS')
epds_results = compare_diagnoses(epds_avg, 'EPDS')
epds_results['Symptom'] = 'EPDS'
print(epds_results)


In [ ]:
effect_size_df = pd.concat([
    stress_results,
    mood_results,
    anh_results,
    dep_results,
    anx_results,
    mpas_results,
    epds_results
], ignore_index=True)
effect_size_df

if coh == [3]:
    effect_size_df.to_excel(OUTPUT_TABLES / "val_clust_cohens.xlsx", index=False)
elif coh == [1, 2]:
    effect_size_df.to_excel(OUTPUT_TABLES / "disc_clust_cohens.xlsx", index=False)

effect_size_df


In [ ]:
effect_size_df['Comparison'] = effect_size_df['Group1'] + ' vs ' + effect_size_df['Group2']

effect_size_df = effect_size_df[[
    'Symptom', 'Comparison', 'Group1', 'Group2', 'Cohens_d', 'CI_lower', 'CI_upper'
]]

stress_avg['Symptom'] = 'Stress'
mood_avg['Symptom'] = 'Mood'
anh_avg['Symptom'] = 'Anhedonia'
dep_avg['Symptom'] = 'Depression'
anx_avg['Symptom'] = 'Anxiety'
mpas_avg['Symptom'] = 'MPAS'
epds_avg['Symptom'] = 'EPDS'

stress_avg = stress_avg.rename(columns={'Stress': 'Value'})
mood_avg = mood_avg.rename(columns={'Mood': 'Value'})
anh_avg = anh_avg.rename(columns={'Anhedonia': 'Value'})
dep_avg = dep_avg.rename(columns={'Depression': 'Value'})
anx_avg = anx_avg.rename(columns={'Anxiety': 'Value'})
mpas_avg = mpas_avg.rename(columns={'MPAS': 'Value'})
epds_avg = epds_avg.rename(columns={'EPDS': 'Value'})

long_avg_df = pd.concat([
    stress_avg,
    mood_avg,
    anh_avg,
    dep_avg,
    anx_avg,
    mpas_avg,
    epds_avg
], ignore_index=True)


In [ ]:
sub = long_avg_df[long_avg_df['Symptom'] == 'Stress']
sub


### Plot effect sizes and participant averages

Show seven measures. The violin order includes HC C1, HC C2, AD C2, and PPD C2. No multiplicity adjustment is applied to bootstrap intervals.

In [ ]:
custom_palette = {
        'HC C1': '#0072B2',
        'AD C1':'#D55E00',
        'HC C2': '#0072B2',
        'AD C2': '#D55E00',
        'PPD C2': '#009E73'
    }

symptom_list = ['Stress', 'Mood', 'EPDS', 'Anhedonia', 'Anxiety', 'Depression', 'MPAS']
n_symptoms = len(symptom_list)

fig = plt.figure(figsize=(6, 2 * n_symptoms))
gs = fig.add_gridspec(n_symptoms, 2, width_ratios=[1.1, 1.5], wspace=0.4, hspace=0.9)
effect_size_df['Symptom'] = pd.Categorical(effect_size_df['Symptom'], categories=symptom_list, ordered=True)
effect_size_df

max_comparisons = effect_size_df.groupby('Symptom').size().max()
shared_y_pos = np.arange(max_comparisons)

for i, symptom in enumerate(symptom_list):
    symptom_data = effect_size_df[effect_size_df['Symptom'] == symptom].copy()
    symptom_data = symptom_data.sort_values(by='Comparison').reset_index(drop=True)

    n_rows = len(symptom_data)
    y_pos = shared_y_pos[:n_rows]
    cohens_d = symptom_data['Cohens_d']
    ci_lowers = cohens_d - symptom_data['CI_lower']
    ci_uppers = symptom_data['CI_upper'] - cohens_d

    if symptom == 'Stress':
        row_title = 'Stress Self-Assessment'
    elif symptom == 'Mood':
        row_title = 'Mood Self-Assessment'
    elif symptom in ['Anhedonia', 'Anxiety', 'Depression']:
        row_title = f"{symptom} (EPDS)"
    else:
        row_title = symptom

    ax_left = fig.add_subplot(gs[i, 0])
    ax_left.errorbar(cohens_d, y_pos, xerr=[ci_lowers, ci_uppers], fmt='o', capsize=4, color='black')
    ax_left.axvline(0, linestyle='--', color='black', lw=2)
    ax_left.set_yticks(y_pos)
    ax_left.set_yticklabels(symptom_data['Comparison'])
    ax_left.invert_yaxis()
    ax_left.set_xlabel("Cohen's d (95% CI)")
    ax_left.set_xlim(-4, 4)
    ax_left.set_ylim(max_comparisons - 0.5, -0.5)
    ax_left.grid(True, axis='x', color='lightgrey', linestyle='--', linewidth=0.7)
    sns.despine(ax=ax_left, left=True, bottom=False)

    ax_left.set_title(row_title, pad=20, ha='center')

    print(sub['Group'].unique())

    ax_right = fig.add_subplot(gs[i, 1])
    sub = long_avg_df[long_avg_df['Symptom'] == symptom]

    group_order = ['HC C1', 'HC C2',  'AD C2', 'PPD C2']
    sns.violinplot(
        data=sub,
        x='Group',
        y='Value',
        hue='Group',
        palette=custom_palette,
        ax=ax_right,
        inner='box',
        linewidth=1.2,
        order=group_order,
        )

    ax_right.set_ylabel(f"{symptom} Score")
    ax_right.set_xlabel('')
    ax_right.tick_params(axis='x')
    ax_right.tick_params(axis='y')
    ax_right.grid(True, axis='y', linestyle='--', color='lightgrey', linewidth=0.7)
    ax_right.grid(False, axis='x')

plt.tight_layout()
plt.subplots_adjust(top=0.95)

if coh == [3]:
    aux_name = 'val'
elif coh == [1, 2]:
    aux_name = 'disc'
for ext in ['png', 'pdf', 'tiff']:
        save_path = OUTPUT_PLOTS / f"{aux_name}_clustered_cohens_violin.{ext}"
        fig.savefig(save_path, dpi=600, bbox_inches='tight', format=ext)

plt.show()


## 4. Categorical risk-factor forest plots

Load both cohorts’ exported relative risks, harmonize display names, exclude the listed factors, and outer-join by variable. Estimates refer to HC at IDX=1 versus HC at IDX=0 in the source analysis. Stars initially come from the exported unadjusted p-values.

In [ ]:
summary_df_cat_disc = pd.read_excel(OUTPUT_FILES / f'rr_cat_[1, 2]_statanalysis.xlsx')
summary_df_cat_val = pd.read_excel(OUTPUT_FILES / f'rr_cat_[3]_statanalysis.xlsx')

summary_df_cat_val
summary_df_cat_val


In [ ]:
import pandas as pd

summary_df_cat_disc = pd.read_excel(OUTPUT_FILES / f'rr_cat_[1, 2]_statanalysis.xlsx')
summary_df_cat_val = pd.read_excel(OUTPUT_FILES / f'rr_cat_[3]_statanalysis.xlsx')

rename_dict = {
    "Highest degree of education": "Education",
    "Income": "Annual income, thousand €",
    "Baby blues": "Baby Blues",
    "Marital status": "Married",
    "Breastfeeding T0": "Breastfeeding at childbirth",
    "Breastfeeding T4": "Breastfeeding at 12 weeks pp",
    "Gender of the child": "Gender of the baby",
    "Birth-related psychological and physical traumas": "Subjective experience of traumatic childbirth",
    "Support at home t0": "Support at home at childbirth",
    "Support at home t4": "Support at home at 12 weeks pp",
    "Psychiatric history binary":"Psychiatric history",
    "Age": "Age, years",
    "Total amount of children": "Total number of children",
    "Stressful life events": "Stressful life events (number)",
    "Weight (gramm)": "Birthweight of the child, g",
    "Height (cm)": "Height, cm",
    "Rejection T0": "Rejection towards the baby T0",
    "Rejection T1": "Rejection towards the baby T1",
    "Rejection T2": "Rejection towards the baby T2",
    "Rejection T3": "Rejection towards the baby T3",
    "Mother care": "Care from own mother",
    "Father care": "Care from own father",
    "Mother overprotection": "Overprotection by own mother",
    "Father overprotection": "Overprotection by own father",
    'Relocation to other ward': 'Relocation of the baby'
}

def rename_values_in_column(df, column, mapping, df_name):
    """Rename matching values in place and print the changes."""
    if column not in df.columns:
        print(f"Column '{column}' not found in {df_name} DataFrame.")
        return df

    renamed = []
    for old_val in df[column]:
        if old_val in mapping:
            renamed.append(mapping[old_val])
            print(f"{df_name}: {old_val} → {mapping[old_val]}")
        else:
            renamed.append(old_val)
    df[column] = renamed
    return df

summary_df_cat_disc = rename_values_in_column(summary_df_cat_disc, "Variable", rename_dict, "Discovery")
summary_df_cat_val = rename_values_in_column(summary_df_cat_val, "Variable", rename_dict, "Validation")


In [ ]:
remove_values = [
    "Completed professional education",
    "SLE: Personal",
    "SLE: Family",
    "Iron supplementation",
    "Family status",
    "Eisenpräparat",
    "In vivo fertilization"
]

def remove_rows_by_value(df, column, values_to_remove, df_name):
    """Exclude listed values and return a table with a reset index."""
    if column not in df.columns:
        print(f"Column '{column}' not found in {df_name} DataFrame.")
        return df

    before = len(df)
    df = df[~df[column].isin(values_to_remove)].reset_index(drop=True)
    removed = before - len(df)

    print(f"{df_name}: Removed {removed} rows where '{column}' had values {values_to_remove}")
    return df

summary_df_cat_disc = remove_rows_by_value(summary_df_cat_disc, "Variable", remove_values, "Discovery")
summary_df_cat_val = remove_rows_by_value(summary_df_cat_val, "Variable", remove_values, "Validation")


In [ ]:
summary_df_cat_disc


In [ ]:
summary_df_cat_val


In [ ]:
for df in [summary_df_cat_disc, summary_df_cat_val]:
    df['RR'] = pd.to_numeric(df['RR'], errors='coerce')
    df['CI Lower'] = pd.to_numeric(df['CI Lower'], errors='coerce')
    df['CI Upper'] = pd.to_numeric(df['CI Upper'], errors='coerce')

merged = pd.merge(
    summary_df_cat_disc,
    summary_df_cat_val,
    on='Variable',
    suffixes=('_disc', '_val'),
    how='outer',
    indicator=True
)

both = merged[merged['_merge'] == 'both']
only_disc = merged[merged['_merge'] == 'left_only']
only_val = merged[merged['_merge'] == 'right_only']
ordered = pd.concat([both, only_disc, only_val], ignore_index=True)
ordered = ordered[::1].reset_index(drop=True)


In [ ]:
ordered['Variable'] = ordered['Variable'].replace({
    'in vitro fertilization2': 'In vitro fertilization',
    'Eisenpräparat': 'Iron supplementation',
        'Eisenpräparat': 'Iron supplementation',
        'Psychiatric history binary': 'Psychiatric history',
        'Psychiatric diagnosis in previous pregancy':'Psychiatric diagnosis in previous pregnancy'

})

ordered


Point positions are capped at 3 while the stored interval bounds remain unchanged. This can produce invalid negative error lengths if the capped point lies outside its interval; the plotting calculation is preserved.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.patheffects as path_effects
from matplotlib.lines import Line2D

color_discovery = '#0072B2'
color_validation = '#D55E00'

y_offset = 0.20
x_star_offset = -0.1
max_rr_display = 3.0

fig, ax = plt.subplots(figsize=(7, 7))
y_positions = np.arange(len(ordered))
y_labels = ordered['Variable'].tolist()

for i, row in ordered.iterrows():
    y = y_positions[i]
    if pd.notna(row['RR_val']):
        rr = min(row['RR_val'], max_rr_display)
        ci_low = row['CI Lower_val']
        ci_up = row['CI Upper_val']
        y_val = y - y_offset

        ax.errorbar(
            x=rr, y=y_val,
            xerr=[[rr - ci_low], [ci_up - rr]],
            fmt='s', color=color_validation, capsize=3, linewidth=0.9,
            label=""
        )

        sig_val = row.get('Significance_val')
        if isinstance(sig_val, str) and sig_val in ['*', '**']:
            txt = ax.text(
                ci_low - 0.05, y_val - x_star_offset,
                sig_val,
                color=color_validation,
                fontsize=20,
                fontweight='bold',
                va='center',
                ha='right'
            )
            txt.set_path_effects([
                path_effects.Stroke(linewidth=1.2, foreground='white'),
                path_effects.Normal()
            ])

for i, row in ordered.iterrows():
    y = y_positions[i]
    if pd.notna(row['RR_disc']):
        rr = min(row['RR_disc'], max_rr_display)
        ci_low = row['CI Lower_disc']
        ci_up = row['CI Upper_disc']
        y_disc = y + y_offset

        ax.errorbar(
            x=rr, y=y_disc,
            xerr=[[rr - ci_low], [ci_up - rr]],
            fmt='o', color=color_discovery, capsize=3, linewidth=0.9,
            label=""
        )

        sig_disc = row.get('Significance_disc')
        if isinstance(sig_disc, str) and sig_disc in ['*', '**']:
            txt = ax.text(
                ci_low - 0.05, y_disc - x_star_offset,
                sig_disc,
                color=color_discovery,
                fontsize=20,
                fontweight='bold',
                va='center',
                ha='right'
            )
            txt.set_path_effects([
                path_effects.Stroke(linewidth=1.2, foreground='white'),
                path_effects.Normal()
            ])

    ax.hlines(
    y=y, xmin=0, xmax=max_rr_display + 1.5,
    color='lightgrey', linestyle=':', linewidth=0.8, alpha=0.7, zorder=0
)

ax.axvline(x=1.0, linestyle='--', color='grey', lw=1.5)
ax.set_yticks(y_positions)
ax.set_yticklabels(y_labels)
ax.set_xlabel("Relative Risk (RR)")
ax.set_xlim(0, max_rr_display + 1.5)
ax.invert_yaxis()

legend_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=color_discovery, markersize=8, label='Discovery'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor=color_validation, markersize=8, label='Validation'),
]
ax.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=2, frameon=False)

ax.grid(True, axis='x', linestyle='--', color='lightgrey', linewidth=0.7)
ax.grid(False, axis='y')

plt.tight_layout(pad=1.5)

filename_base = OUTPUT_PLOTS / "relative_risk"
fig.savefig(f"{filename_base}.png", dpi=600, bbox_inches='tight', format='png')
fig.savefig(f"{filename_base}.pdf", dpi=600, bbox_inches='tight', format='pdf')
fig.savefig(f"{filename_base}.tiff", dpi=600, bbox_inches='tight', format='tiff')

plt.show()


Apply BH correction only to nonmissing p-values for variables present exclusively in validation. The input p-values were rounded in the risk-factor export. This printed analysis does not update the preceding relative-risk figure.

In [ ]:
from statsmodels.stats.multitest import multipletests

only_val = ordered[ordered['_merge'] == 'right_only']

only_val_pvals = only_val[['Variable', 'P-value_val']].dropna()

rejected, pvals_corrected, _, _ = multipletests(only_val_pvals['P-value_val'], alpha=0.05, method='fdr_bh')

only_val_pvals['FDR_corrected_p'] = pvals_corrected
only_val_pvals['FDR_significant'] = rejected

significant_vars_val = only_val_pvals[only_val_pvals['FDR_significant']]

print("Significant (FDR-corrected) variables only in validation cohort:")
print(significant_vars_val[['Variable', 'P-value_val', 'FDR_corrected_p']])


## 5. Continuous risk-factor forest plots

Load Cohen’s d summaries, harmonize names, exclude the listed factors, and align cohorts by variable. Positive d indicates a higher mean in HC at IDX=1. The existing point-position cap at 3 is retained.

In [ ]:
summary_df_cont_disc = pd.read_excel(OUTPUT_FILES / f'cd_cont_[1, 2]_statanalysis.xlsx')
summary_df_cont_val = pd.read_excel(OUTPUT_FILES / f'cd_cont_[3]_statanalysis.xlsx')

summary_df_cont_disc = summary_df_cont_disc.rename(columns={'ci_lower': 'CI Lower', 'ci_upper': 'CI Upper'})
summary_df_cont_disc = summary_df_cont_disc.rename(columns={'Cohen\'s d': 'Cohens d'})
summary_df_cont_val = summary_df_cont_val.rename(columns={'Cohen\'s d': 'Cohens d'})

rename_dict_coh = {
    "Highest degree of education": "Education",
    "Income": "Annual income, thousand €",
    "Baby blues": "Baby Blues",
    "Marital status": "Married",
    "Breastfeeding T0": "Breastfeeding at childbirth",
    "Breastfeeding T4": "Breastfeeding at 12 weeks pp",
    "Gender of the child": "Gender of the baby",
    "Birth-related psychological and physical traumas": "Subjective experience of traumatic childbirth",
    "Support at home t0": "Support at home at childbirth",
    "Support at home t4": "Support at home at 12 weeks pp",
    "Psychiatric history binary":"Psychiatric history",
    "Age": "Age, years",
    "Total amount of children": "Total number of children",
    "Stressful life events": "Stressful life events (number)",
    "Weight (gramm)": "Birthweight of the child, g",
    "Height (cm)": "Height, cm",
    "Ablehnung T4": "Rejection towards the baby T4",
    "Ablehnung T1": "Rejection towards the baby T1",
    "Ablehnung T2": "Rejection towards the baby T2",
    "Ablehnung T3": "Rejection towards the baby T3",
    "Mutter Care": "Care from own mother",
    "Vater Care": "Care from own father",
    "Mutter Overprotection": "Overprotection by own mother",
    "Vater Overprotection": "Overprotection by own father",
    "RLS_Wert": "RLS score",
    "PMS_value": "PMS score"
}

summary_df_cont_disc = rename_values_in_column(summary_df_cont_disc, "Variable", rename_dict_coh, "Discovery")
summary_df_cont_val = rename_values_in_column(summary_df_cont_val, "Variable", rename_dict_coh, "Validation")

summary_df_cont_disc = remove_rows_by_value(summary_df_cont_disc, "Variable", remove_values, "Discovery")
summary_df_cont_val = remove_rows_by_value(summary_df_cont_val, "Variable", remove_values, "Validation")
summary_df_cont_disc


In [ ]:
summary_df_cont_val


In [ ]:
for df in [summary_df_cont_disc, summary_df_cont_val]:
    pass

merged = pd.merge(
    summary_df_cont_disc,
    summary_df_cont_val,
    on='Variable',
    suffixes=('_disc', '_val'),
    how='outer',
    indicator=True
)

merged['P-value_disc'] = pd.to_numeric(merged['P-value_disc'], errors='coerce')
merged['P-value_val'] = pd.to_numeric(merged['P-value_val'], errors='coerce')
merged = merged[~merged['Variable'].isin(['HAMILTON', 'StressQ T4'])]

both = merged[merged['_merge'] == 'both']
only_disc = merged[merged['_merge'] == 'left_only']
only_val = merged[merged['_merge'] == 'right_only']
ordered = pd.concat([both, only_disc, only_val], ignore_index=True)
ordered = ordered[::1].reset_index(drop=True)

color_discovery = '#0072B2'
color_validation = '#D55E00'
ordered['Variable'] = ordered['Variable'].replace({
    'RLS_Wert': 'RLS score',
    'PMS_value': 'PMS score',
        'Ablehnung T1': 'Rejection T1',
        'Ablehnung T2': 'Rejection T2',
        'Ablehnung T3': 'Rejection T3',
        'Ablehnung T4': 'Rejection T4',
        'Mutter Care': 'Mother care',
        'Vater Care': 'Father care',
        'Mutter Overprotection': 'Mother overprotection',
        'Vater Overprotection': 'Father overprotection'

})

ordered


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.patheffects as path_effects
from matplotlib.lines import Line2D

color_discovery = '#0072B2'
color_validation = '#D55E00'

y_offset = 0.2
x_star_offset = -0.05
max_rr_display = 3.0

fig, ax = plt.subplots(figsize=(7, 7))
y_positions = np.arange(len(ordered))
y_labels = []

plotted_legend = {'Discovery': False, 'Validation': False}

for i, row in ordered.iterrows():
    y = y_positions[i]
    y_labels.append(row['Variable'])

    if pd.notna(row['Cohens d_disc']):
        rr = min(row['Cohens d_disc'], max_rr_display)
        ci_low = row['CI Lower_disc']
        ci_up = row['CI Upper_disc']

        y_disc = y + y_offset
        ax.errorbar(
            x=rr, y=y_disc,
            xerr=[[rr - ci_low], [ci_up - rr]],
            fmt='o', color=color_discovery, capsize=4,
            label='Discovery' if not plotted_legend['Discovery'] else ""
        )
        plotted_legend['Discovery'] = True

        if row['Cohens d_disc'] > max_rr_display:
            ax.text(max_rr_display + 0.1, y_disc, f"{row['Cohens d_disc']:.2f}", color=color_discovery, va='center')

        sig_disc = row.get('Significance_disc')
        if isinstance(sig_disc, str) and sig_disc in ['*', '**']:
            txt = ax.text(
                ci_low + x_star_offset, y_disc,
                sig_disc,
                color=color_discovery,
                fontsize=16,
                fontweight='bold',
                va='center',
                ha='right'
            )
            txt.set_path_effects([
                path_effects.Stroke(linewidth=1.2, foreground='white'),
                path_effects.Normal()
            ])

    if pd.notna(row['Cohens d_val']):
        rr = min(row['Cohens d_val'], max_rr_display)
        ci_low = row['CI Lower_val']
        ci_up = row['CI Upper_val']

        y_val = y - y_offset
        ax.errorbar(
            x=rr, y=y_val,
            xerr=[[rr - ci_low], [ci_up - rr]],
            fmt='s', color=color_validation, capsize=6,
            label='Validation' if not plotted_legend['Validation'] else ""
        )
        plotted_legend['Validation'] = True

        if row['Cohens d_val'] > max_rr_display:
            ax.text(max_rr_display + 0.1, y_val, f"{row['Cohens d_val']:.2f}", color=color_validation, va='center')

        sig_val = row.get('Significance_val')
        if isinstance(sig_val, str) and sig_val in ['*', '**']:
            txt = ax.text(
                ci_low + x_star_offset, y_val,
                sig_val,
                color=color_validation,
                fontsize=16,
                fontweight='bold',
                va='center',
                ha='right'
            )
            txt.set_path_effects([
                path_effects.Stroke(linewidth=1.2, foreground='white'),
                path_effects.Normal()
            ])

ax.axvline(x=0.0, linestyle='--', color='black', lw=1)
ax.set_yticks(y_positions)
ax.set_yticklabels(y_labels)
ax.set_xlabel("Cohen's d")
ax.invert_yaxis()

legend_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=color_discovery, markersize=8, label='Discovery'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor=color_validation, markersize=8, label='Validation')
]
ax.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=2, frameon=False)

ax.grid(True, axis='x', linestyle='--', color='lightgrey', linewidth=0.7)
ax.grid(False, axis='y')
plt.tight_layout(pad=1.5)

filename_base = OUTPUT_PLOTS / "cohensd_risk"

fig.savefig(f"{filename_base}.png", dpi=600, bbox_inches='tight', format='png')
fig.savefig(f"{filename_base}.pdf", dpi=600, bbox_inches='tight', format='pdf')
fig.savefig(f"{filename_base}.tiff", dpi=600, bbox_inches='tight', format='tiff')

plt.show()

from statsmodels.stats.multitest import multipletests

only_val = ordered[ordered['_merge'] == 'right_only']

only_val_pvals = only_val[['Variable', 'P-value_val']].dropna()

rejected, pvals_corrected, _, _ = multipletests(only_val_pvals['P-value_val'], alpha=0.05, method='fdr_bh')

only_val_pvals['FDR_corrected_p'] = pvals_corrected
only_val_pvals['FDR_significant'] = rejected

significant_vars_val = only_val_pvals[only_val_pvals['FDR_significant']]

print("Significant (FDR-corrected) variables only in validation cohort:")
print(significant_vars_val[['Variable', 'P-value_val', 'FDR_corrected_p']])


### Final annotated version

Apply BH correction only to validation-only variables and display `**` for those significant after correction. Other annotations use imported significance fields; a discovery star for stressful life events is hard-coded. These symbols do not share a universal p-value threshold. This cell overwrites the preceding continuous-factor figure files.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.patheffects as path_effects
from matplotlib.lines import Line2D
from statsmodels.stats.multitest import multipletests

color_discovery = '#0072B2'
color_validation = '#D55E00'

y_offset = 0.2
x_star_offset = -0.02
max_rr_display = 3.0

only_val = ordered[ordered['_merge'] == 'right_only']
only_val_pvals = only_val[['Variable', 'P-value_val']].dropna()
rejected, pvals_corrected, _, _ = multipletests(only_val_pvals['P-value_val'], alpha=0.05, method='fdr_bh')
only_val_pvals['FDR_corrected_p'] = pvals_corrected
only_val_pvals['FDR_significant'] = rejected
significant_vars_val = only_val_pvals[only_val_pvals['FDR_significant']]
fdr_significant_vars = set(significant_vars_val['Variable'])

fig, ax = plt.subplots(figsize=(7, 7))
y_positions = np.arange(len(ordered))
y_labels = []
plotted_legend = {'Discovery': False, 'Validation': False}

for i, row in ordered.iterrows():
    y = y_positions[i]
    y_labels.append(row['Variable'])

    if pd.notna(row['Cohens d_disc']):
        rr = min(row['Cohens d_disc'], max_rr_display)
        ci_low = row['CI Lower_disc']
        ci_up = row['CI Upper_disc']
        y_disc = y + y_offset

        ax.errorbar(
            x=rr, y=y_disc,
            xerr=[[rr - ci_low], [ci_up - rr]],
            fmt='o', color=color_discovery, capsize=3, linewidth=0.8,
            label='Discovery' if not plotted_legend['Discovery'] else ""
        )
        plotted_legend['Discovery'] = True

        if row['Cohens d_disc'] > max_rr_display:
            ax.text(max_rr_display + 0.1, y_disc, f"{row['Cohens d_disc']:.2f}", color=color_discovery, va='center')

        if row['Variable'] == 'Stressful life events (number)':
            sig_disc = '*'
        else:
            sig_disc = row.get('Significance')
        if isinstance(sig_disc, str) and sig_disc in ['*', '**']:
            txt = ax.text(
                ci_low + x_star_offset, y_disc,
                sig_disc,
                color=color_discovery,
                fontsize=16,
                fontweight='bold',
                va='center',
                ha='right'
            )
            txt.set_path_effects([
                path_effects.Stroke(linewidth=1.2, foreground='white'),
                path_effects.Normal()
            ])

    if pd.notna(row['Cohens d_val']):
        rr = min(row['Cohens d_val'], max_rr_display)
        ci_low = row['CI Lower_val']
        ci_up = row['CI Upper_val']
        y_val = y - y_offset

        ax.errorbar(
            x=rr, y=y_val,
            xerr=[[rr - ci_low], [ci_up - rr]],
            fmt='s', color=color_validation, capsize=3, linewidth=0.8,
            label='Validation' if not plotted_legend['Validation'] else ""
        )
        plotted_legend['Validation'] = True

        if row['Cohens d_val'] > max_rr_display:
            ax.text(max_rr_display + 0.1, y_val, f"{row['Cohens d_val']:.2f}", color=color_validation, va='center')

        sig_val = '**' if row['Variable'] in fdr_significant_vars else row.get('Significance_val')
        if isinstance(sig_val, str) and sig_val in ['*', '**']:
            txt = ax.text(
                ci_low + x_star_offset, y_val,
                sig_val,
                color=color_validation,
                fontsize=20,
                fontweight='bold',
                va='center',
                ha='right'
            )
            txt.set_path_effects([
                path_effects.Stroke(linewidth=1.2, foreground='white'),
                path_effects.Normal()
                ])

    xmin, xmax = -1.2, 1.2
    for y in np.arange(len(ordered)):
        ax.hlines(y, xmin, xmax, color='lightgrey', linestyle=':', linewidth=0.8, alpha=0.7, zorder=0)

ax.axvline(x=0.0, linestyle='--', color='grey', lw=1)
ax.set_yticks(y_positions)
ax.set_yticklabels(y_labels)
ax.set_xlabel("Cohen's d")
ax.invert_yaxis()

legend_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=color_discovery, markersize=8, label='Discovery'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor=color_validation, markersize=8, label='Validation')
]
ax.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=2, frameon=False)

ax.grid(True, axis='x', linestyle='--', color='lightgrey', linewidth=0.7)
ax.grid(False, axis='y')
plt.tight_layout(pad=1.5)

filename_base = OUTPUT_PLOTS / "cohensd_risk"
fig.savefig(f"{filename_base}.png", dpi=800, bbox_inches='tight', format='png')
fig.savefig(f"{filename_base}.pdf", dpi=800, bbox_inches='tight', format='svg')
fig.savefig(f"{filename_base}.tiff", dpi=800, bbox_inches='tight', format='tiff')

plt.show()

print("Significant (FDR-corrected) variables only in validation cohort:")
print(significant_vars_val[['Variable', 'P-value_val', 'FDR_corrected_p']])


## 6. Combine the risk-factor figures

Read the exported TIFFs, resize the second image to the first image’s height if needed, and place them side by side.

In [ ]:
from PIL import Image

img2 = Image.open(OUTPUT_PLOTS / 'cohensd_risk.tiff')
img1 = Image.open(OUTPUT_PLOTS / 'relative_risk.tiff')

if img1.height != img2.height:
    img2 = img2.resize((img2.width, img1.height))

total_width = img1.width + img2.width
max_height = max(img1.height, img2.height)
merged_img = Image.new("RGB", (total_width, max_height), color=(255, 255, 255))

merged_img.paste(img1, (0, 0))
merged_img.paste(img2, (img1.width, 0))

merged_img.show()

merged_img.save(OUTPUT_PLOTS / 'merged_riskfactors.tiff', dpi=(600, 600), compression="tiff_deflate")


The labeled composite adds A/B and fixed numeric annotations “26.5” and “10.7” at manually specified positions. They are not calculated from the current results and require review if the input data change.

In [ ]:
from PIL import Image, ImageDraw, ImageFont

img2 = Image.open(OUTPUT_PLOTS / 'cohensd_risk.tiff')
img1 = Image.open(OUTPUT_PLOTS / 'relative_risk.tiff')

if img1.height != img2.height:
    img2 = img2.resize((img2.width, img1.height))

total_width = img1.width + img2.width
max_height = max(img1.height, img2.height)
merged_img = Image.new("RGB", (total_width, max_height), color=(255, 255, 255))

merged_img.paste(img1, (0, 0))
merged_img.paste(img2, (img1.width, 0))

draw = ImageDraw.Draw(merged_img)

try:
    font_big = ImageFont.truetype("arial.ttf", 150)
    font_num = ImageFont.truetype("arial.ttf", 70)
except:
    font_big = font_num = ImageFont.load_default()

padding_x = 100
padding_y = 90
draw.text((padding_x, padding_y), "A", fill="black", font=font_big)
draw.text((img1.width + padding_x, padding_y), "B", fill="black", font=font_big)

num1 = "26.5"
num2 = "10.7"

color_num1 = "#D55E00"
color_num2 = "#0072B2"

bbox1 = draw.textbbox((0, 0), num1, font=font_num)
bbox2 = draw.textbbox((0, 0), num2, font=font_num)
h1 = bbox1[3] - bbox1[1]
h2 = bbox2[3] - bbox2[1]
gap = 130

total_h = h1 + h2 + gap

x = total_width - 6300
y = (max_height - total_h) // 2 - 127

draw.text((x, y), num1, fill=color_num1, font=font_num)
draw.text((x, y + h1 + gap), num2, fill=color_num2, font=font_num)

merged_img.save(OUTPUT_PLOTS / 'merged_riskfactors_labeled.tiff', dpi=(600, 600), compression="tiff_lzw")
merged_img.show()
